# Static Baselines — Classical ML

Train track-level emotion classifiers on aggregated spectral features.

**Design notes:**
- Splits are created at **track_id** level (no window leakage).
- Emotion quadrant thresholds are fit on **train tracks only**.
- Metrics and confusion matrices are saved to `results/metrics/` and `reports/figures/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_project_root, load_configs, resolve_path
from src.features.spectral_features import load_static_features
from src.data.splits import (
    assert_no_track_leakage,
    create_track_split_table,
    save_track_splits,
    split_track_ids,
)
from src.training.train_static import train_and_evaluate_static_models

configs = load_configs(PROJECT_ROOT)
root = get_project_root(PROJECT_ROOT)

## 1. Load features and labels

In [2]:
features_df = load_static_features(configs)
labels_df = pd.read_parquet(resolve_path(root, configs["paths"]["processed"]["static_labels"]))

print(f"Features: {features_df.shape}")
print(f"Labels: {labels_df.shape}")

Features: (1802, 187)
Labels: (1802, 11)


## 2. Create track-level splits (no leakage)

In [3]:
split_df = create_track_split_table(
    labels_df,
    track_col="song_id",
    label_col="emotion_quadrant",
    configs=configs,
)
save_track_splits(split_df, configs)

split_counts = split_df["split"].value_counts()
split_counts

split
train    1261
test      271
val       270
Name: count, dtype: int64

In [4]:
# Verify no track appears in multiple splits
splits_dict = {
    split: split_df.loc[split_df["split"] == split, "song_id"].tolist()
    for split in ("train", "val", "test")
}
assert_no_track_leakage(splits_dict)
print("No track leakage detected.")

No track leakage detected.


## 3. Train and evaluate classical models

In [5]:
summary_df = train_and_evaluate_static_models(
    configs=configs,
    features_df=features_df,
    labels_df=labels_df,
    split_df=split_df,
)

summary_df.sort_values(["model_name", "eval_split"])

c:\Users\athen\miniconda3\envs\music-emotion-recognition\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,model_name,task_type,feature_type,target_type,eval_split,train_size,val_size,test_size,accuracy,macro_f1,weighted_f1,mae,rmse,r2,comments
9,gradient_boosting,static,spectral_aggregated,emotion_quadrant,test,1261,270,271,0.571956,0.379244,0.517763,None,None,None,Thresholds fit on train tracks only; splits by...
8,gradient_boosting,static,spectral_aggregated,emotion_quadrant,val,1261,270,271,0.611111,0.410433,0.549406,None,None,None,Thresholds fit on train tracks only; splits by...
3,knn,static,spectral_aggregated,emotion_quadrant,test,1261,270,271,0.564576,0.421810,0.539531,None,None,None,Thresholds fit on train tracks only; splits by...
2,knn,static,spectral_aggregated,emotion_quadrant,val,1261,270,271,0.551852,0.423505,0.533473,None,None,None,Thresholds fit on train tracks only; splits by...
1,logistic_regression,static,spectral_aggregated,emotion_quadrant,test,1261,270,271,0.564576,0.427884,0.545204,None,None,None,Thresholds fit on train tracks only; splits by...
0,logistic_regression,static,spectral_aggregated,emotion_quadrant,val,1261,270,271,0.544444,0.384512,0.518591,None,None,None,Thresholds fit on train tracks only; splits by...
13,mlp,static,spectral_aggregated,emotion_quadrant,test,1261,270,271,0.579336,0.408228,0.540874,None,None,None,Thresholds fit on train tracks only; splits by...
12,mlp,static,spectral_aggregated,emotion_quadrant,val,1261,270,271,0.603704,0.446655,0.567235,None,None,None,Thresholds fit on train tracks only; splits by...
7,random_forest,static,spectral_aggregated,emotion_quadrant,test,1261,270,271,0.608856,0.376117,0.525322,None,None,None,Thresholds fit on train tracks only; splits by...
6,random_forest,static,spectral_aggregated,emotion_quadrant,val,1261,270,271,0.614815,0.377374,0.530224,None,None,None,Thresholds fit on train tracks only; splits by...


## 4. Results location

- Metrics JSON: `results/metrics/static_{model}_{split}_metrics.json`
- Summary table: `results/metrics/static_baselines_summary.csv`
- Confusion matrices: `reports/figures/static_{model}_{split}_confusion_matrix.png`